# 16 UMAP / t-SNE Feature Visualization

Check whether SDD learns representations with better class separability than baseline via 2D projection.

- UMAP: `pip install umap-learn` required
- t-SNE: always available through sklearn

In [ ]:
import torch
from src.experiments import load_cfg, deep_update

# ── GPU Auto-Detection ───────────────────────────────────────
n_gpu = torch.cuda.device_count()
print(f"Detected number of GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory // 1024**3} GB")
if n_gpu == 0:
    print("  (No GPU — running on CPU)")


In [ ]:
from src.experiments import (
    load_cfg, deep_update, build_loaders,
    build_trainer_from_checkpoint,
    run_umap_comparison, run_tsne_comparison,
)

CIFAR10_CLASSES = ["airplane","automobile","bird","cat","deer",
                   "dog","frog","horse","ship","truck"]

cfg_b = load_cfg("configs/cifar10.yaml")
cfg_b = deep_update(cfg_b, {
    "run_name": "cifar10_baseline_mse_only",
    "sdd": {"enabled": False, "lambda_distill": 0.0,
            "use_centering": False, "use_sharpening": False,
            "use_gating": False, "use_projection_head": False},
})
cfg_sdd = load_cfg("configs/cifar10.yaml")
cfg_sdd = deep_update(cfg_sdd, {"run_name": "cifar10_full_sdd"})

trainer_b   = build_trainer_from_checkpoint(cfg_b,   "outputs/checkpoints/cifar10_baseline_mse_only_last.pt")
trainer_sdd = build_trainer_from_checkpoint(cfg_sdd, "outputs/checkpoints/cifar10_full_sdd_last.pt")
_, test_loader = build_loaders(cfg_sdd)

trainers = {"Baseline (MSE only)": trainer_b, "Full SDD": trainer_sdd}
print("✓ Checkpoints loaded")


In [ ]:
# t-SNE (always available)
tsne_path = run_tsne_comparison(
    trainers, test_loader,
    feature_layer="bottleneck",
    max_samples=2000,
    save_path="outputs/figures/tsne_comparison.png",
    class_names=CIFAR10_CLASSES,
)
print(f"t-SNE saved: {tsne_path}")


In [ ]:
# UMAP (umap-learn required)
try:
    umap_path = run_umap_comparison(
        trainers, test_loader,
        feature_layer="bottleneck",
        max_samples=2000,
        save_path="outputs/figures/umap_comparison.png",
        class_names=CIFAR10_CLASSES,
    )
    print(f"UMAP saved: {umap_path}")
except ImportError:
    print("umap-learn is not installed. Install it with: pip install umap-learn")


In [ ]:
# Layer-wise UMAP comparison for the Full SDD model
import torch, matplotlib.pyplot as plt, matplotlib.cm as cm
from src.evaluation.feature_extract import extract_features

layers = ["bottleneck", "skip1", "skip2", "decoder1"]

try:
    import umap
    fig, axes = plt.subplots(1, len(layers), figsize=(6 * len(layers), 5))
    for ax, layer in zip(axes, layers):
        feats, labels = extract_features(
            trainer_sdd._unwrap(trainer_sdd.model),
            test_loader, trainer_sdd.device, feature_layer=layer
        )
        idx   = torch.randperm(feats.shape[0])[:2000]
        f_np  = feats[idx].numpy()
        l_np  = labels[idx].numpy()
        emb   = umap.UMAP(n_components=2, random_state=42).fit_transform(f_np)
        colors = cm.get_cmap("tab10", 10)
        for c in range(10):
            mask = l_np == c
            ax.scatter(emb[mask, 0], emb[mask, 1], s=5, alpha=0.6,
                       color=colors(c), label=CIFAR10_CLASSES[c])
        ax.set_title(f"layer: {layer}", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
    axes[-1].legend(markerscale=2, fontsize=7, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.suptitle("UMAP by feature layer (Full SDD)", fontsize=13)
    plt.tight_layout()
    plt.savefig("outputs/figures/umap_layer_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("umap-learn is not installed.")
